# Notebook 1B: Your First Complete Kernel on A100

This notebook checks the environment, displays the source, builds it, and runs `vector_add`. Run it from inside the repository.


In [ ]:
from pathlib import Path
import shutil
import subprocess

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "CMakeLists.txt").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the cuda-kernels-a100-beginners repository")

ROOT = find_repo_root(Path.cwd())
print("repo:", ROOT)
required = ("nvidia-smi", "nvcc", "cmake")
missing = [command for command in required if shutil.which(command) is None]
assert not missing, f"Missing required commands: {missing}"
gpus = subprocess.run(["nvidia-smi", "-L"], text=True, capture_output=True, check=True)
print(gpus.stdout)


If `nvidia-smi` or `nvcc` returns `None`, stop and repair the environment. Do not simulate a successful CUDA run.


In [ ]:
source = (ROOT / "lessons/06_vector_add.cu").read_text()
print(source)


Find these parts in the code: index calculation, bounds check, three device allocations, two H2D copies, kernel launch, D2H copy, and result verification.


In [ ]:
subprocess.run([
    "cmake", "-S", str(ROOT), "-B", str(ROOT / "build"),
    "-DCMAKE_BUILD_TYPE=Release", "-DCMAKE_CUDA_ARCHITECTURES=80"
], check=True)
subprocess.run(["cmake", "--build", str(ROOT / "build"), "--target", "00_device_query", "06_vector_add", "-j"], check=True)


In [ ]:
device = subprocess.run([str(ROOT / "build/00_device_query")], text=True, capture_output=True, check=True)
print(device.stdout)
device_zero = device.stdout.split("Device 1:", 1)[0]
assert "Device 0:" in device_zero
assert "A100" in device_zero, "CUDA device 0 is not identified as an A100; set CUDA_VISIBLE_DEVICES"
assert "compute capability: 8.0" in device_zero, "CUDA device 0 is not compute capability 8.0"


In [ ]:
result = subprocess.run([str(ROOT / "build/06_vector_add")], text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0
assert "PASS vector_add" in result.stdout


## Questions

1. Why is `if (i < n)` required even when `n` is large?
2. What does the CUDA-event time include, and what does it exclude?
3. Change `n` to a value that is not divisible by 256 and verify that the result remains correct.
